In [0]:
%run ./1_config

nyc_taxi


In [0]:
df_bronze = spark.read.table(t_bronze_trips)

catalog : nyc_taxi
schemas : landing / bronze / silver / gold
landing volume : /Volumes/nyc_taxi/landing/files


In [0]:
import pyspark.sql.functions as f

### Convert pickup datetime column into a timestamp

In [0]:
df = df_bronze.withColumn("pickup_ts", f.to_timestamp("tpep_pickup_datetime"))

display(df.select("tpep_pickup_datetime", "pickup_ts").limit(10))
df.printSchema()

tpep_pickup_datetime,pickup_ts
2024-01-01T00:57:55.000,2024-01-01T00:57:55.000Z
2024-01-01T00:03:00.000,2024-01-01T00:03:00.000Z
2024-01-01T00:17:06.000,2024-01-01T00:17:06.000Z
2024-01-01T00:36:38.000,2024-01-01T00:36:38.000Z
2024-01-01T00:46:51.000,2024-01-01T00:46:51.000Z
2024-01-01T00:54:08.000,2024-01-01T00:54:08.000Z
2024-01-01T00:49:44.000,2024-01-01T00:49:44.000Z
2024-01-01T00:30:40.000,2024-01-01T00:30:40.000Z
2024-01-01T00:26:01.000,2024-01-01T00:26:01.000Z
2024-01-01T00:28:08.000,2024-01-01T00:28:08.000Z


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- _ingest_time: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- pickup_ts: timestamp (nullable

### Extract only the date part from pickup_ts into a new column

In [0]:
df = df.withColumn("pickup_date", f.to_date("pickup_ts"))

display(df.select("pickup_ts", "pickup_date").limit(10))

pickup_ts,pickup_date
2024-01-01T00:57:55.000Z,2024-01-01
2024-01-01T00:03:00.000Z,2024-01-01
2024-01-01T00:17:06.000Z,2024-01-01
2024-01-01T00:36:38.000Z,2024-01-01
2024-01-01T00:46:51.000Z,2024-01-01
2024-01-01T00:54:08.000Z,2024-01-01
2024-01-01T00:49:44.000Z,2024-01-01
2024-01-01T00:30:40.000Z,2024-01-01
2024-01-01T00:26:01.000Z,2024-01-01
2024-01-01T00:28:08.000Z,2024-01-01


### Extract the hour from pickup_ts into a new column

In [0]:
df = df.withColumn("pickup_hour", f.hour("pickup_ts"))

display(df.select("pickup_ts", "pickup_hour").limit(10))

pickup_ts,pickup_hour
2024-01-01T00:57:55.000Z,0
2024-01-01T00:03:00.000Z,0
2024-01-01T00:17:06.000Z,0
2024-01-01T00:36:38.000Z,0
2024-01-01T00:46:51.000Z,0
2024-01-01T00:54:08.000Z,0
2024-01-01T00:49:44.000Z,0
2024-01-01T00:30:40.000Z,0
2024-01-01T00:26:01.000Z,0
2024-01-01T00:28:08.000Z,0


### Calculate trip duration in minutes (dropoff time - pickup time)

In [0]:
df = df.withColumn("dropoff_ts", f.to_timestamp("tpep_dropoff_datetime"))

display(df.select("tpep_dropoff_datetime", "dropoff_ts").limit(10))

tpep_dropoff_datetime,dropoff_ts
2024-01-01T01:17:43.000,2024-01-01T01:17:43.000Z
2024-01-01T00:09:36.000,2024-01-01T00:09:36.000Z
2024-01-01T00:35:01.000,2024-01-01T00:35:01.000Z
2024-01-01T00:44:56.000,2024-01-01T00:44:56.000Z
2024-01-01T00:52:57.000,2024-01-01T00:52:57.000Z
2024-01-01T01:26:31.000,2024-01-01T01:26:31.000Z
2024-01-01T01:15:47.000,2024-01-01T01:15:47.000Z
2024-01-01T00:58:40.000,2024-01-01T00:58:40.000Z
2024-01-01T00:54:12.000,2024-01-01T00:54:12.000Z
2024-01-01T00:29:16.000,2024-01-01T00:29:16.000Z


In [0]:
df = df.withColumn("trip_duration_min",
                   f.round(
                       (f.col("dropoff_ts").cast("long") - f.col("pickup_ts").cast("long")) / 60, 2)
)

display(df.select("pickup_ts", "dropoff_ts", "trip_duration_min").limit(10))

pickup_ts,dropoff_ts,trip_duration_min
2024-01-01T00:57:55.000Z,2024-01-01T01:17:43.000Z,19.8
2024-01-01T00:03:00.000Z,2024-01-01T00:09:36.000Z,6.6
2024-01-01T00:17:06.000Z,2024-01-01T00:35:01.000Z,17.92
2024-01-01T00:36:38.000Z,2024-01-01T00:44:56.000Z,8.3
2024-01-01T00:46:51.000Z,2024-01-01T00:52:57.000Z,6.1
2024-01-01T00:54:08.000Z,2024-01-01T01:26:31.000Z,32.38
2024-01-01T00:49:44.000Z,2024-01-01T01:15:47.000Z,26.05
2024-01-01T00:30:40.000Z,2024-01-01T00:58:40.000Z,28.0
2024-01-01T00:26:01.000Z,2024-01-01T00:54:12.000Z,28.18
2024-01-01T00:28:08.000Z,2024-01-01T00:29:16.000Z,1.13


### Calculate average speed in miles per hour
### speed = distance / duration (hours)

In [0]:
df = df.withColumn("avg_speed",
                   f.round(
                       f.when(f.col("trip_duration_min") > 0,
                              f.col("trip_distance") / (f.col("trip_duration_min") / 60)), 2)
)

display(df.select("trip_distance", "trip_duration_min", "avg_speed").limit(10))

trip_distance,trip_duration_min,avg_speed
1.72,19.8,5.21
1.8,6.6,16.36
4.7,17.92,15.74
1.4,8.3,10.12
0.8,6.1,7.87
4.7,32.38,8.71
10.82,26.05,24.92
3.0,28.0,6.43
5.44,28.18,11.58
0.04,1.13,2.12


### convert VendorID, PULocationID, DOLocationID, passenger_count into integer

In [0]:
df = df.withColumn("VendorID", f.col("VendorID").cast("int"))
df = df.withColumn("PULocationID", f.col("PULocationID").cast("int"))
df = df.withColumn("DOLocationID", f.col("DOLocationID").cast("int"))
df = df.withColumn("passenger_count", f.col("passenger_count").cast("int"))

df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- _ingest_time: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- pickup_ts: timestamp (nulla

### airport_fee: replace null into 0 
lit stands for literal — converts plain fixed value into a Column type so that Spark functions which expect a Column can use it.

In [0]:
df = df.withColumn("airport_fee", f.coalesce(f.col("airport_fee"), f.lit(0)))

display(df.select("airport_fee").limit(5))

airport_fee
0.0
0.0
0.0
0.0
0.0


### congestion_surcharge: replace null to 0 

In [0]:
df = df.withColumn("congestion_surcharge", f.coalesce(f.col("congestion_surcharge"), f.lit(0)))

display(df.select("congestion_surcharge").limit(5))

congestion_surcharge
2.5
2.5
2.5
2.5
2.5


### new col: tip_pct
Formula: tip_pct = (tip_amount / fare_amount) * 100                                                               
Condition: fare_amount must be greater than 0

In [0]:
df = df.withColumn("tip_pct", 
                   f.round(
                       f.when(f.col("fare_amount") > 0,
                              (f.col("tip_amount") / f.col("fare_amount")) * 100), 2)
)

display(df.select("fare_amount", "tip_amount", "tip_pct").limit(10))

fare_amount,tip_amount,tip_pct
17.7,0.0,0.0
10.0,3.75,37.5
23.3,3.0,12.88
10.0,2.0,20.0
7.9,3.2,40.51
29.6,6.9,23.31
45.7,10.0,21.88
25.4,0.0,0.0
31.0,0.0,0.0
3.0,0.0,0.0


### new col: airport_trip_valid
Check If pickup location or drop location is 1, 132, or 138, it is considered an airport trip

In [0]:
df = df.withColumn("airport_trip_valid",
                   f.when(
                       f.col("PULocationID").isin(1, 132, 138) | f.col("DOLocationID").isin(1, 132, 138),
                       True)
                   .otherwise(False)
)

display(df.select("PULocationID", "DOLocationID", "airport_trip_valid").limit(10))

PULocationID,DOLocationID,airport_trip_valid
186,79,false
140,236,false
236,79,false
79,211,false
211,148,false
148,141,false
138,181,true
246,231,false
161,261,false
113,113,false


### if payment_type missing then replace with 5 

In [0]:
df = df.withColumn("payment_type",
                   f.when(f.col("payment_type").isNull(), 5).otherwise(f.col("payment_type"))
)

display(df.select("payment_type").limit(10))

payment_type
2
1
1
1
1
1
1
2
2
2


### check clean and unclean data 
Mark a row as unclean if:                                                                      
pickup_ts is null, OR
dropoff_ts is null, OR
dropoff_ts is earlier than pickup_ts                                                
If none of these three are true → the row is clean.

In [0]:
df_checked = df.withColumn("is_unclean",
                           (f.col("pickup_ts").isNull()) |
                           (f.col("dropoff_ts").isNull()) |
                           (f.col("dropoff_ts") < f.col("pickup_ts"))
)

display(df_checked.select("pickup_ts", "dropoff_ts", "is_unclean").limit(10))

pickup_ts,dropoff_ts,is_unclean
2024-01-01T00:57:55.000Z,2024-01-01T01:17:43.000Z,false
2024-01-01T00:03:00.000Z,2024-01-01T00:09:36.000Z,false
2024-01-01T00:17:06.000Z,2024-01-01T00:35:01.000Z,false
2024-01-01T00:36:38.000Z,2024-01-01T00:44:56.000Z,false
2024-01-01T00:46:51.000Z,2024-01-01T00:52:57.000Z,false
2024-01-01T00:54:08.000Z,2024-01-01T01:26:31.000Z,false
2024-01-01T00:49:44.000Z,2024-01-01T01:15:47.000Z,false
2024-01-01T00:30:40.000Z,2024-01-01T00:58:40.000Z,false
2024-01-01T00:26:01.000Z,2024-01-01T00:54:12.000Z,false
2024-01-01T00:28:08.000Z,2024-01-01T00:29:16.000Z,false


In [0]:
df_trips_clean = df_checked.filter(f.col("is_unclean") == False)
df_trips_unclean = df_checked.filter(f.col("is_unclean") == True)

print("clean rows:", df_trips_clean.count())
print("unclean rows:", df_trips_unclean.count())

clean rows: 2964568
unclean rows: 56
